# Simulation Code T = 2

## Import libraries

In [ ]:
from utils.dynamicRieszFunctions import estimateDynamicRiesz_all
from utils.dynamicRieszFunctions import estimateDynamicRiesz
import torch
import pandas as pd
import time
from torch.distributions import Normal

/Users/wisserutgers/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## Estimation Settings

In [2]:
lasso_cv_settings = {
    'b_degree' : 1,
    'cv_folds' : 5,
    'random_state' : 42
}

lasso_a_settings = {
    'lambda_val' : 0,
    'beta_start' : None,
    'D_LB' : 0,
    'D_add' : 0.2,
    'c1' : "CV",
    'c2' : 0.1,
    'tol' : 1e-5,
    'max_iter' : 100,
    'b_degree' : 1,
    'control' : {'maxIter': 1000, 'optTol': 1e-5, 'zeroThreshold': 1e-6}
}

lasso_f_settings = {
    'lambda_val' : 0,
    'beta_start' : None,
    'D_LB' : 0,
    'D_add' : 0.2,
    'c1' :  "CV",
    'c2' : 0.1,
    'tol' : 1e-5,
    'max_iter' : 100,
    'b_degree' : 1,
    'control' : {'maxIter': 1000, 'optTol': 1e-5, 'zeroThreshold': 1e-6}
}

rf_a_settings = {
    'poly_degree' : 0,
    'l2' : 0,
    'n_estimators' : 100,
    'criterion' : "mse",
    'max_depth' : None,
    'min_samples_split' : 10,
    'min_samples_leaf' : 5,
    'min_weight_fraction_leaf' : 0.,
    'min_var_fraction_leaf' : None,
    'min_var_leaf_on_val' : False,
    'max_features' : "auto",
    'min_impurity_decrease' : 0.,
    'max_samples' : .45,
    'min_balancedness_tol' : .45,
    'honest' : True,
    'inference' : True,
    'fit_intercept' : True,
    'subforest_size' : 4,
    'n_jobs' : -1,
    'random_state' : None,
    'verbose' : 0,
    'warm_start' : False
}
rf_f_settings = {
    'poly_degree' : 1, # 1 or 2?
    'l2' : 0,
    'n_estimators' : 100,
    'criterion' : "mse",
    'max_depth' : None,
    'min_samples_split' : 10,
    'min_samples_leaf' : 5,
    'min_weight_fraction_leaf' : 0.,
    'min_var_fraction_leaf' : None,
    'min_var_leaf_on_val' : False,
    'max_features' : "auto",
    'min_impurity_decrease' : 0.,
    'max_samples' : .45,
    'min_balancedness_tol' : .45,
    'honest' : True,
    'inference' : True,
    'fit_intercept' : True,
    'subforest_size' : 4,
    'n_jobs' : -1,
    'random_state' : None,
    'verbose' : 0,
    'warm_start' : False
}

net_a_settings = {
    'test_split' : 0,
    'learner_lr' : 1e-4,
    'learner_l2' : 1e-3,
    'learner_l1' : 0,
    'n_epochs' : 100,
    'earlystop_rounds' : 20,
    'earlystop_delta' : 1e-3,
    'bs' : 64,
    'optimizer' : 'adam',
    'warm_start' : False,
    'logger' : None,
    'model_dir' : '.',
    'device' : torch.cuda.current_device() if torch.cuda.is_available() else None,
    'n_hidden' : 100,
    'drop_prob' : 0,
    'degree' : 2,
    'interaction_only' : True,
    'n_common' : 200,
    'act_func' : 'elu'
}

net_f_settings = {
    'test_split' : 0,
    'learner_lr' : 1e-4,
    'learner_l2' : 1e-3,
    'learner_l1' : 0,
    'n_epochs' : 100,
    'earlystop_rounds' : 20,
    'earlystop_delta' : 1e-3,
    'bs' : 64,
    'optimizer' : 'adam',
    'warm_start' : False,
    'logger' : None,
    'model_dir' : '.',
    'device' : torch.cuda.current_device() if torch.cuda.is_available() else None,
    'n_hidden' : 100,
    'drop_prob' : 0,
    'degree' : 2,
    'interaction_only' : True,
    'n_common' : 200,
    'act_func' : 'elu'
}


## Generate data 

##### Simulation settings:

In [3]:
torch.manual_seed(123);

# Parameters
N = 1000
tmax = 100

# Higher dimensions = more sparsity. Minimum is dimX1 = dimX2 = 3
dimX1 = 5
dimX2 = 5

##### Define treatment probability function

In [4]:
# Bounds (only for truncated distributions)
lower = 0.10
upper = 0.90

# Define logistic function
def logistic(x):
    return torch.exp(x) / (1 + torch.exp(x))

# Define a truncated logistic function
def truncated_logistic(x):
    return lower + (upper - lower) * logistic(x)

# Simple nonlinear probability (from adversarial Riesz paper)
def truncated_adv(x):
    return lower + (upper - lower) * (x > 0)

# Simple nonlinear probability (from adversarial Riesz paper)
def double_nonlinear(x):
    return lower + (upper - lower) * ((x < - 1/2) + (x < 1/2))

In [5]:
treatment_probability_func = truncated_logistic

##### Coefficients

In [6]:
# # Treatment probability period 1
# beta_pi1_0 = 0 # Intercept 
# beta_pi1_X1 = torch.tensor([1] + [0] * (dimX1 - 1), dtype=torch.float32).reshape(-1,1) # Effect of X1 

# # Treatment probability period 2
# beta_pi2_0_0 = -0.2 # Intercept if untreated in period 1
# beta_pi2_0_X2 = torch.tensor([1] + [0] * (dimX2 - 1), dtype=torch.float32).reshape(-1,1) # Effect of X2 if untreated in period 1
# beta_pi2_1_0 = 0.2 # Intercept if treated in period 1
# beta_pi2_1_X2 = torch.tensor([1] + [0] * (dimX2 - 1), dtype=torch.float32).reshape(-1,1) # Effect of X2 if treated in period 1

# Treatment probability period 1
beta_pi1_0 = 0 # Intercept 
beta_pi1_X1 = torch.tensor([1] + [0] * (dimX1 - 1), dtype=torch.float32).reshape(-1,1) # Effect of X1 

# Treatment probability period 2
beta_pi2_0_0 = 0 # Intercept if untreated in period 1
beta_pi2_0_X2 = torch.tensor([0.5] + [0] * (dimX2 - 1), dtype=torch.float32).reshape(-1,1) # Effect of X2 if untreated in period 1
beta_pi2_1_0 = 0 # Intercept if treated in period 1
beta_pi2_1_X2 = torch.tensor([1] + [0] * (dimX2 - 1), dtype=torch.float32).reshape(-1,1) # Effect of X2 if treated in period 1

# Outcome
c1 = 2.2
c2 = 1.2
c12 = 0.5

beta1 = 1.2
beta2 = 0.8

delta1 = 1
delta2 = 1
delta12 = 0.5


##### Simulate data

In [7]:
# Period 1
X1_all = torch.randn(N * tmax, dimX1) # Normally distributed X1 covariates
pi1_all = treatment_probability_func(beta_pi1_0 + X1_all @ beta_pi1_X1).reshape(-1,1) # Generated treatment probablities
D1_all = torch.bernoulli(pi1_all).int().reshape(-1,1) # Generate treatment assignment period 1

# Period 2
delta1_all = torch.randn(N * tmax,1) # Change from X1 to X2 if treated in period 1
delta2_all = torch.randn(N * tmax, dimX2) # Change from X1 to X2 always

X2_all = X1_all + D1_all * (1 + delta1_all) + delta2_all # Period 2 covariates
X2_1_all = X1_all + 1 + delta1_all + delta2_all # Period 2 covariates if treated in period 1
X2_0_all = X1_all + delta2_all # Period 2 covariates if untreated in period 1

pi2_0_all = treatment_probability_func(beta_pi2_0_0 + X2_0_all @ beta_pi2_0_X2) # Treatment probability if untreated in period 1
pi2_1_all = treatment_probability_func(beta_pi2_1_0 + X2_1_all @ beta_pi2_1_X2) # Treatment probability if treated in period 1
pi2_all = (1 - D1_all) * pi2_0_all + D1_all * pi2_1_all # Actual treatment probability

D2_all = torch.bernoulli(pi2_all).int() # Generate treatment assignment period 1

# Outcome
g_all = (c1 * D1_all + c2 * D2_all + c12 * D1_all * D2_all + 
        beta1 * (X1_all[:,:1] > 0) + beta2 * (X2_all[:,:1] > 0) + 
        # X1_all[:,:1] + delta1 * D1_all * X1_all[:,:1] + delta2 * D2_all * X2_all[:,:1] + delta12 * D1_all * D2_all * X2_all[:,:1])
        delta1 * D1_all * X1_all[:,:1] + delta2 * D2_all * X2_all[:,:1] + delta12 * D1_all * D2_all * X2_all[:,:1])
zeta_all = torch.randn(N * tmax,1) # Noise to outcome
Y_all = g_all + zeta_all

#### True values:

In [8]:
normal_dist1 = Normal(1, torch.sqrt(torch.tensor(3)))
cdf_value1 = 1 - normal_dist1.cdf(torch.tensor(0))


normal_dist0 = Normal(0, torch.sqrt(torch.tensor(2)))
cdf_value0 = 1 -  normal_dist0.cdf(-1 - X1_all[:,:1])


normal_dist0 = Normal(0, torch.sqrt(torch.tensor(2)))
cdf_value01 = 1 -  normal_dist0.cdf(- X1_all[:,:1])

mu2_1_all = (c1 * 1 + c2 * 1 + c12 * 1 * 1 + 
        beta1 * (X1_all[:,:1] > 0) + beta2 * (X2_1_all[:,:1] > 0) + 
        delta1 * 1 * X1_all[:,:1] + delta2 * 1 * X2_1_all[:,:1] + delta12 * 1 * 1 * X2_1_all[:,:1])

mu2_0_all = (c1 * 0 + c2 * 0 + c12 * 0 * 0 + 
        beta1 * (X1_all[:,:1] > 0) + beta2 * (X2_0_all[:,:1] > 0) + 
        delta1 * 0 * X1_all[:,:1] + delta2 * 0 * X2_0_all[:,:1] + delta12 * 0 * 0  * X2_0_all[:,:1])

mu1_1_all = (c1 * 1 + c2 * 1 + c12 * 1 * 1 + 
        beta1 * (X1_all[:,:1] > 0) + beta2 * cdf_value0 + 
        delta1 * 1 * X1_all[:,:1] + delta2 * 1 * (1 + X1_all[:,:1]) + delta12 * 1 * 1  * (1 + X1_all[:,:1]))

mu1_0_all = (c1 * 0 + c2 * 0 + c12 * 0 * 0 + 
        beta1 * (X1_all[:,:1] > 0) + beta2 * (X1_all[:,:1] > 0) + 
        delta1 * 0 * X1_all[:,:1] + delta2 * 0 * X1_all[:,:1] + delta12 * 0 * 0 * X1_all[:,:1])

print(mu2_1_all.mean(), mu2_0_all.mean(), mu1_1_all.mean(), mu1_0_all.mean())

theta0 = 0.5 * beta1 + 0.5 * beta2
theta1 = (c1 + c2 + c12 + 
        0.5 * beta1 + cdf_value1 * beta2 +
        (delta2 + delta12) * 1)

print(theta0, theta1)


tensor(6.5661) tensor(0.9966) tensor(6.5614) tensor(0.9969)
1.0 tensor(6.5745)


## Estimation:

#### Estimation settings

In [9]:
folds = 4

time0 = time.time()

#### Estimation 

In [10]:
pred_theta = torch.zeros(tmax,18)
pred_sig = torch.zeros(tmax,18)

RR1 = torch.zeros(N,tmax,18)
RR2 = torch.zeros(N,tmax,18)
f1 = torch.zeros(N,tmax,18)
f2 = torch.zeros(N,tmax,18)

for t in range(0,tmax):

    # Get data for current iteration
    X1, X2 = X1_all[(t) * N : (t+1) * N, :], X2_all[(t) * N : (t+1) * N, :]
    D1, D2 = D1_all[(t) * N : (t+1) * N], D2_all[(t) * N : (t+1) * N]
    Y = Y_all[(t) * N : (t+1) * N]

    X = torch.hstack((X1,X2))
    X_index = torch.tensor([X1.shape[1]-1,X1.shape[1]+X2.shape[1]-1])
    D = torch.hstack((D1,D2))

    # Set counterfactual of interest
    d = 1 * torch.ones(D.shape) 

    #---------------------------------------------------------------------------
    # Estimator 1: Oracle
    pi1, pi2_0, pi2_1  = pi1_all[(t) * N : (t+1) * N], pi2_0_all[(t) * N : (t+1) * N], pi2_1_all[(t) * N : (t+1) * N]
    mu2_1, mu2_0, mu1_1, mu1_0 = mu2_1_all[(t) * N : (t+1) * N], mu2_0_all[(t) * N : (t+1) * N], mu1_1_all[(t) * N : (t+1) * N], mu1_0_all[(t) * N : (t+1) * N]
    if d[0,0] == 1:
        RR2[:,t,:1], RR1[:,t,:1] = D1 * D2 / (pi1 * pi2_1), D1 / pi1 
        theta_hat = (RR2[:,t,:1] * Y - (RR1[:,t,:1] - 1) * mu1_1   - (RR2[:,t,:1] - RR1[:,t,:1]) * mu2_1)
        pred_theta[t,0] = torch.mean(theta_hat)
        pred_sig[t,0] = torch.sqrt(torch.mean((theta_hat - torch.mean(theta_hat) ) ** 2))
    elif d[0,0] == 0:
        RR2[:,t,:1], RR1[:,t,:1] = (1 - D1) * (1 - D2) / ((1 - pi1  ) * (1 - pi2_0  )), (1 - D1) / (1 - pi1)
        theta_hat = (RR2[:,t,:1] * Y - (RR1[:,t,:1] - 1) * mu1_0   - (RR2[:,t,:1] - RR1[:,t,:1]) * mu2_0)
        pred_theta[t,0] = torch.mean(theta_hat)
        pred_sig[t,0] = torch.sqrt(torch.mean((theta_hat - torch.mean(theta_hat) ) ** 2))

    #---------------------------------------------------------------------------
    # Estimator 2: Bradic
    # try:
    #     bradic_result = estimateBradic(Y, X, D, X_index, folds)
    #     if d[0,0] == 1:
    #         pred_theta[t,1], pred_sig[t,1] = bradic_result[7], torch.sqrt(torch.tensor(bradic_result[10]))
    #     elif d[0,0] == 0:
    #         pred_theta[t,1], pred_sig[t,1] = bradic_result[13], torch.sqrt(torch.tensor(bradic_result[16]))
    # except:
    #     pred_theta[t,1], pred_sig[t,1] = 1000,1000
    #---------------------------------------------------------------------------
    # # Estimator 3: LASSO, RF, Net

    pred_theta[t,2:], pred_sig[t,2:], RR1[:,t,2:], RR2[:,t,2:] , f1[:,t,2:], f2[:,t,2:] = estimateDynamicRiesz_all(Y, X, D, d, X_index, folds)

    # pred_theta[t,3], pred_sig[t,3], RR1[:,t,3:4], RR2[:,t,3:4], f1[:,t,3:4], f2[:,t,3:4] = estimateDynamicRiesz(Y, X, D, d, X_index, folds,
    #                                                                  method_a = "RF", rf_a_settings = rf_a_settings,
    #                                                                     method_f = "RF", rf_f_settings = rf_f_settings)

    if t % 10 == 0:
        print("Time until iteration ",t, ": ", time.time() - time0)

print("Finished. Total time: ", time.time() - time0)


Time until iteration  0 :  23.732796907424927


KeyboardInterrupt: 

## Compute statistics

#### Get true value

In [ ]:
true_value = theta1

#### Compute statistics

In [ ]:
bias = torch.mean(pred_theta[:t+1,:] - true_value, 0)
rmse = torch.sqrt(torch.mean( (pred_theta[:t+1,:] - true_value) ** 2, 0))

ub = pred_theta[:t+1,:] + 1.96 * pred_sig[:t+1,:] / torch.sqrt(torch.tensor(N))
lb = pred_theta[:t+1,:] - 1.96 * pred_sig[:t+1,:] / torch.sqrt(torch.tensor(N))

coverage = torch.mean( ( (true_value >= lb) & (true_value <= ub) ).float() , 0 )
interval_length = torch.mean(ub - lb,0)

# Create a DataFrame
data = {
    "Method": ["Oracle", "Bradic", "LASSO_LASSO", "LASSO_RF", "LASSO_Net", "LASSO_RF1", "RF_LASSO", "RF_RF", "RF_Net", "RF_RF1", "Net_LASSO", "Net_RF", "Net_Net", "Net_RF1", "RF0_LASSO", "RF0_RF", "RF0_Net", "RF0_RF1"],
    "Bias": bias.ravel(),
    "RMSE": rmse.ravel(),
    "Coverage": coverage.ravel(),
    "Interval Length": interval_length.ravel()
}

#### Print results

In [ ]:
df = pd.DataFrame(data)
print(df)

         Method      Bias      RMSE  Coverage  Interval Length
0        Oracle  0.018391  0.176320  0.947368         0.649767
1        Bradic -6.574519  6.574519  0.000000         0.000000
2   LASSO_LASSO -6.574519  6.574519  0.000000         0.000000
3      LASSO_RF  0.165661  1.132932  0.289474         0.591954
4     LASSO_Net -6.574519  6.574519  0.000000         0.000000
5     LASSO_RF1 -6.574519  6.574519  0.000000         0.000000
6      RF_LASSO -6.574519  6.574519  0.000000         0.000000
7         RF_RF -6.574519  6.574519  0.000000         0.000000
8        RF_Net -6.574519  6.574519  0.000000         0.000000
9        RF_RF1 -6.574519  6.574519  0.000000         0.000000
10    Net_LASSO -6.574519  6.574519  0.000000         0.000000
11       Net_RF -6.574519  6.574519  0.000000         0.000000
12      Net_Net -6.574519  6.574519  0.000000         0.000000
13      Net_RF1 -6.574519  6.574519  0.000000         0.000000
14    RF0_LASSO -6.574519  6.574519  0.000000         0

In [ ]:
pred_theta[:,0:5]

tensor([[6.6085, 0.0000, 0.0000, 6.8793, 0.0000],
        [6.6333, 0.0000, 0.0000, 6.8853, 0.0000],
        [6.6575, 0.0000, 0.0000, 6.9990, 0.0000],
        [6.5993, 0.0000, 0.0000, 6.9724, 0.0000],
        [6.7132, 0.0000, 0.0000, 7.1115, 0.0000],
        [6.6790, 0.0000, 0.0000, 6.9884, 0.0000],
        [6.5378, 0.0000, 0.0000, 6.8825, 0.0000],
        [6.4156, 0.0000, 0.0000, 6.7457, 0.0000],
        [6.8206, 0.0000, 0.0000, 7.0344, 0.0000],
        [6.8186, 0.0000, 0.0000, 7.0343, 0.0000],
        [6.4220, 0.0000, 0.0000, 6.8704, 0.0000],
        [6.8560, 0.0000, 0.0000, 7.1032, 0.0000],
        [6.6648, 0.0000, 0.0000, 6.9622, 0.0000],
        [6.3610, 0.0000, 0.0000, 6.5899, 0.0000],
        [6.4094, 0.0000, 0.0000, 6.6401, 0.0000],
        [6.5185, 0.0000, 0.0000, 6.8153, 0.0000],
        [6.5205, 0.0000, 0.0000, 6.9255, 0.0000],
        [6.1413, 0.0000, 0.0000, 6.5910, 0.0000],
        [6.6620, 0.0000, 0.0000, 6.9218, 0.0000],
        [6.2877, 0.0000, 0.0000, 6.6291, 0.0000],


In [ ]:
# check_t = 0
# Compare RR:
# pd.DataFrame(torch.hstack((RR1[:,check_t,0:1], RR1[:,check_t,2:3], RR1[:,check_t,3:4], RR1[:,check_t,4:5]))).head(50)
# pd.DataFrame(torch.hstack((RR2[:,check_t,0:1], RR2[:,check_t,2:3], RR2[:,check_t,3:4], RR2[:,check_t,4:5]))).head(50)


In [ ]:
D